In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")
payer_table = dbutils.widgets.get("payer_table")
billto_table = dbutils.widgets.get("billto_table")
cobactive_tbl = dbutils.widgets.get("cobactive_tbl")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customfile_src AS
SELECT
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  AcctNbr AS AcctNbr,
  Company AS Company,
  Practice AS Practice,
  BayadaRegion AS BayadaRegion,
  BayadaDivision AS BayadaDivision,
  BayadaOfficeAbbreviation AS BayadaOfficeAbbreviation,
  CompanyTaxID AS CompanyTaxID,
  ReimbursementTeam AS ReimbursementTeam,
  PayerSourceNumber AS PayerSourceNumber,
  PayerSourceProgramName AS PayerSourceProgramName,
  BillingPeriodOrFrequency AS BillingPeriodOrFrequency,
  EVVRequirements AS EVVRequirements,
  EVVAggregator AS EVVAggregator,
  EVVCodesInScope AS EVVCodesInScope,
  COBPrimaryInsurance AS COBPrimaryInsurance,
  ReferralID AS ReferralID,
  PrimarySubscriberIDNumber AS PrimarySubscriberIDNumber,
  COBStatus AS COBStatus,
  COBType AS COBType,
  COBEffDate AS COBEffDate,
  COBLevel AS COBLevel,
  COBDenialReason AS COBDenialReason,
  COBSplitDecision AS COBSplitDecision,
  LimitedBenefit AS LimitedBenefit,
  BenefitDate AS BenefitDate,
  BillHoldReason AS BillHoldReason,
  BillHoldDate AS BillHoldDate,
  SourceSystemKey AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  cob_base AS (
      SELECT *
      FROM {cobactive_tbl}
      WHERE COB_Eff_Date IS NOT NULL
        AND COBTerm_Date IS NOT NULL
  ),
  cob_dates_fixed AS (
      SELECT
          *, -- COB Recd Date: MM-dd-yyyy → yyyy-MM-dd
          CONCAT(
              RIGHT(COB_Recd_Date, 4), '-',
              LEFT(COB_Recd_Date, 5)
          ) AS COB_Recd_Date_fmt
      FROM cob_base
  ),
  cob_latest AS (
      SELECT *
      FROM (
          SELECT
              *,
              ROW_NUMBER() OVER (
                  PARTITION BY Client_Id
                  ORDER BY COB_Recd_Date_fmt DESC
              ) AS rnb
          FROM cob_dates_fixed
      )
      WHERE rnb = 1
  ),
  cob_final AS (
      SELECT
          *,
          CONCAT(RIGHT(COB_Eff_Date, 4), '-', LEFT(COB_Eff_Date, 5))  AS COB_Eff_Date_fmt,
          CONCAT(RIGHT(COBTerm_Date, 4), '-', LEFT(COBTerm_Date, 5)) AS COBTerm_Date_fmt
      FROM cob_latest
  ),
  ob_base AS (
      SELECT *
      FROM {source_table} 
      WHERE account_balance != 0 
      AND date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  ob_dates_fixed AS (
      SELECT
          *, -- yyyyMMdd → yyyy-MM-dd
          CONCAT(
              LEFT(first_visit_date_key, 4), '-',
              SUBSTRING(first_visit_date_key, 5, 2), '-',
              RIGHT(first_visit_date_key, 2)
          ) AS First_Visit_Date_fmt,
          CONCAT(
              LEFT(last_visit_date_key, 4), '-',
              SUBSTRING(last_visit_date_key, 5, 2), '-',
              RIGHT(last_visit_date_key, 2)
          ) AS Last_Visit_Date_fmt
      FROM ob_base
  ),
  customfile_cte AS (
      SELECT
          to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
          ofc.OfficeNumber AS FacilityCode,
          CASE
              WHEN UPPER(obd.invoice_number) = 'ADV'
                THEN CONCAT('ADV - ', clt.SourceSystemId)
              ELSE obd.invoice_number
          END AS AcctNbr,
          ofc.Company AS Company,
          ofc.Practice AS Practice,
          ofc.Region AS BayadaRegion,
          ofc.Division AS BayadaDivision,
          ofc.OfficeAbbreviation AS BayadaOfficeAbbreviation,
          NULL AS CompanyTaxID,
          ofc.ReimbursementOfficeName AS ReimbursementTeam,
          py.PayerID AS PayerSourceNumber,
          CASE
              WHEN py.PayorProgram LIKE '%"%' THEN TRIM(REPLACE(py.PayorProgram, '"', ' '))
              WHEN py.PayorProgram LIKE '%*%' THEN TRIM(REPLACE(py.PayorProgram, '*', ' '))
              ELSE py.PayorProgram
          END AS PayerSourceProgramName,
          bt.BillingPeriod AS BillingPeriodOrFrequency,
          NULL AS EVVRequirements,
          NULL AS EVVAggregator,
          NULL AS EVVCodesInScope,
          cob.Payor_Name__Other_Bill_To_ AS COBPrimaryInsurance,
          cob.Referral_Id AS ReferralID,
          cob.Subscriber_Id AS PrimarySubscriberIDNumber,
          cob.COB_Status AS COBStatus,
          cob.COB_Type AS COBType,
          cob.COB_Eff_Date_fmt AS COBEffDate,
          cob.COB_Level AS COBLevel,
          cob.COB_Denial_Reason AS COBDenialReason,
          cob.Split_Decision AS COBSplitDecision,
          cob.Limited_Benefit AS LimitedBenefit,
          cob.Benefit_Date AS BenefitDate,
          cob.Bill_Hold_Reason AS BillHoldReason,
          cob.Bill_Hold_Date AS BillHoldDate,
          0 AS SourceSystemKey
      FROM ob_dates_fixed obd
      LEFT JOIN {office_table} ofc
          ON ofc.OfficeKey = obd.office_key
      LEFT JOIN {payer_table} py
          ON py.PayerKey = obd.payor_key
      LEFT JOIN {billto_table} bt
          ON bt.BillToId = py.PayerID
      LEFT JOIN (
          SELECT
              clt.ClientKey,
              c.*
          FROM cob_final c
          JOIN {client_table} clt
              ON clt.SourceSystemId = c.Client_Id
      ) cob
          ON cob.ClientKey = obd.Client_Key
         AND datediff(obd.First_Visit_Date_fmt, cob.COB_Eff_Date_fmt) < 0
         AND datediff(obd.Last_Visit_Date_fmt,  cob.COBTerm_Date_fmt) > 0
      LEFT JOIN {client_table} clt
          ON clt.ClientKey = obd.client_key
  ),
  customfile_clean AS (
    SELECT 
      *,
      row_number() OVER (
        PARTITION BY AcctNbr, FacilityCode 
        ORDER BY AcctNbr
      ) AS rn
    FROM customfile_cte
  )
  SELECT 
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Company,
    Practice,
    BayadaRegion,
    BayadaDivision,
    BayadaOfficeAbbreviation,
    CompanyTaxID,
    ReimbursementTeam,
    PayerSourceNumber,
    PayerSourceProgramName,
    BillingPeriodOrFrequency,
    EVVRequirements,
    EVVAggregator,
    EVVCodesInScope,
    COBPrimaryInsurance,
    ReferralID,
    PrimarySubscriberIDNumber,
    COBStatus,
    COBType,
    COBEffDate,
    COBLevel,
    COBDenialReason,
    COBSplitDecision,
    LimitedBenefit,
    BenefitDate,
    BillHoldReason,
    BillHoldDate,
    SourceSystemKey
  FROM customfile_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING customfile_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Company = src.Company,
    tgt.Practice = src.Practice,
    tgt.BayadaRegion = src.BayadaRegion,
    tgt.BayadaDivision = src.BayadaDivision,
    tgt.BayadaOfficeAbbreviation = src.BayadaOfficeAbbreviation,
    tgt.CompanyTaxID = src.CompanyTaxID,
    tgt.ReimbursementTeam = src.ReimbursementTeam,
    tgt.PayerSourceNumber = src.PayerSourceNumber,
    tgt.PayerSourceProgramName = src.PayerSourceProgramName,
    tgt.BillingPeriodOrFrequency = src.BillingPeriodOrFrequency,
    tgt.EVVRequirements = src.EVVRequirements,
    tgt.EVVAggregator = src.EVVAggregator,
    tgt.EVVCodesinScope = src.EVVCodesinScope,
    tgt.COBPrimaryInsurance = src.COBPrimaryInsurance,
    tgt.ReferralID = src.ReferralID,
    tgt.PrimarySubscriberIDNumber = src.PrimarySubscriberIDNumber,
    tgt.COBStatus = src.COBStatus,
    tgt.COBType = src.COBType,
    tgt.COBEffDate = src.COBEffDate,
    tgt.COBLevel = src.COBLevel,
    tgt.COBDenialReason = src.COBDenialReason,
    tgt.COBSplitDecision = src.COBSplitDecision,
    tgt.LimitedBenefit = src.LimitedBenefit,
    tgt.BenefitDate = src.BenefitDate,
    tgt.BillHoldReason = src.BillHoldReason,
    tgt.BillHoldDate = src.BillHoldDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Company,
    Practice,
    BayadaRegion,
    BayadaDivision,
    BayadaOfficeAbbreviation,
    CompanyTaxID,
    ReimbursementTeam,
    PayerSourceNumber,
    PayerSourceProgramName,
    BillingPeriodOrFrequency,
    EVVRequirements,
    EVVAggregator,
    EVVCodesinScope,
    COBPrimaryInsurance,
    ReferralID,
    PrimarySubscriberIDNumber,
    COBStatus,
    COBType,
    COBEffDate,
    COBLevel,
    COBDenialReason,
    COBSplitDecision,
    LimitedBenefit,
    BenefitDate,
    BillHoldReason,
    BillHoldDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Company,
    src.Practice,
    src.BayadaRegion,
    src.BayadaDivision,
    src.BayadaOfficeAbbreviation,
    src.CompanyTaxID,
    src.ReimbursementTeam,
    src.PayerSourceNumber,
    src.PayerSourceProgramName,
    src.BillingPeriodOrFrequency,
    src.EVVRequirements,
    src.EVVAggregator,
    src.EVVCodesinScope,
    src.COBPrimaryInsurance,
    src.ReferralID,
    src.PrimarySubscriberIDNumber,
    src.COBStatus,
    src.COBType,
    src.COBEffDate,
    src.COBLevel,
    src.COBDenialReason,
    src.COBSplitDecision,
    src.LimitedBenefit,
    src.BenefitDate,
    src.BillHoldReason,
    src.BillHoldDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)